# DataDriftDetector – Usage Examples

This notebook shows how to use the `DataDriftDetector` class to detect feature drift between a reference dataset and a current dataset.  
It supports:

- **Numeric features**: PSI, KL divergence, KS statistic, JS divergence, Wasserstein distance  
- **Categorical features**: PSI, KL, JS, Wasserstein distance (KS not available)

The detector automatically separates features by type and handles missing columns or incompatible dtypes with warnings.

In [1]:
import numpy as np
import pandas as pd
import warnings

# The class we want to demonstrate
from momo_ml.monitor.data_drift import DataDriftDetector

# For reproducibility
np.random.seed(42)

In [10]:
def create_drift_data(n_ref=1000, n_cur=800, shift_amount=0.5):
    """
    Create reference and current datasets with numeric and categorical features.
    A shift is introduced in the current dataset for demonstration.
    Both DataFrames are generated independently with the specified row counts.
    """
    # Reference data
    ref = pd.DataFrame({
        'age': np.random.normal(35, 10, n_ref),
        'income': np.random.lognormal(mean=10, sigma=0.5, size=n_ref),
        'gender': np.random.choice(['M', 'F'], n_ref, p=[0.5, 0.5]),
        'education': np.random.choice(['HighSchool', 'Bachelor', 'Master'], n_ref, p=[0.4, 0.4, 0.2])
    })

    # Current data – generated independently with distribution shifts
    cur = pd.DataFrame({
        'age': np.random.normal(35 + shift_amount * 10, 10, n_cur),
        'income': np.random.lognormal(mean=10 + np.log(1 + shift_amount), sigma=0.5, size=n_cur),
        'gender': np.random.choice(['M', 'F'], n_cur, p=[0.6, 0.4]),
        'education': np.random.choice(['HighSchool', 'Bachelor', 'Master'], n_cur, p=[0.3, 0.5, 0.2])
    })
    return ref, cur

In [11]:
ref_df, cur_df = create_drift_data(n_ref=2000, n_cur=1500, shift_amount=0.3)

print("Reference shape:", ref_df.shape)
print("Current shape:", cur_df.shape)
ref_df.head()

Reference shape: (2000, 4)
Current shape: (1500, 4)


,age,income,gender,education
0,39.374983,15530.381048,F,Bachelor
1,31.094177,18173.415466,F,Bachelor
2,28.176154,32357.539221,M,Bachelor
3,20.142605,9845.043400,M,Master
4,44.719187,18440.332847,M,Master


##  1. Basic usage (auto‑detect common features)

In [12]:
# Initialize detector without specifying features (uses intersection of columns)
detector = DataDriftDetector(ref_df, cur_df)

print("Auto‑detected features:", detector.features)
print("Numeric features:", detector.numeric_features)
print("Categorical features:", detector.categorical_features)

Auto‑detected features: ['gender', 'age', 'income', 'education']
Numeric features: ['age', 'income']
Categorical features: ['gender', 'education']


## 2. Specifying a list of features (with a missing column)

In [13]:
# Include a feature that doesn't exist in both datasets
specified_features = ['age', 'income', 'gender', 'nonexistent', 'education']

detector2 = DataDriftDetector(ref_df, cur_df, features=specified_features)

print("Used features:", detector2.features)
print("Missing/warned features should have been skipped.")

Used features: ['age', 'income', 'gender', 'education']
Missing/warned features should have been skipped.


E:\git repo\momo_ml\momo_ml\monitor\data_drift.py:58: UserWarning: The following features are missing in one of the datasets and will be skipped: nonexistent
  warnings.warn(msg)


## 3. Numeric vs categorical detection

In [14]:
# Let's see how the detector splits features
detector3 = DataDriftDetector(ref_df, cur_df)
print("Numeric features:", detector3.numeric_features)
print("Categorical features:", detector3.categorical_features)

Numeric features: ['age', 'income']
Categorical features: ['gender', 'education']


## 4. Compute drift for all numeric features individually

In [15]:
numeric_drift = detector3.compute_numeric_drift()

# Display results for a numeric feature, e.g., 'age'
print("Drift metrics for 'age':")
for metric, value in numeric_drift['age'].items():
    if metric == 'ks':
        # ks returns a dict with statistic and pvalue
        print(f"  {metric}: statistic={value['statistic']:.4f}, pvalue={value['pvalue']:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

Drift metrics for 'age':
  psi: 0.0846
  kl: 0.0423
  ks: statistic=0.1273, pvalue=0.0000
  js: 0.0105
  wd: 2.8674


## 5. Compute drift for all categorical features

In [16]:
categorical_drift = detector3.compute_categorical_drift()

print("Drift metrics for 'gender':")
for metric, value in categorical_drift['gender'].items():
    print(f"  {metric}: {value:.4f}")

print("\nDrift metrics for 'education':")
for metric, value in categorical_drift['education'].items():
    print(f"  {metric}: {value:.4f}")

Drift metrics for 'gender':
  psi: 0.0392
  kl: 0.0197
  js: 0.0049
  wd: 0.0985

Drift metrics for 'education':
  psi: 0.0330
  kl: 0.0166
  js: 0.0041
  wd: 0.0825


## 6. Full drift detection (numeric + categorical)

In [17]:
full_results = detector3.compute()

print("=== Numeric Features ===")
for feat, metrics in full_results['numeric_features'].items():
    print(f"\n{feat}:")
    for k, v in metrics.items():
        if k == 'ks':
            print(f"  {k}: stat={v['statistic']:.4f}, p={v['pvalue']:.4f}")
        else:
            print(f"  {k}: {v:.4f}")

print("\n=== Categorical Features ===")
for feat, metrics in full_results['categorical_features'].items():
    print(f"\n{feat}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

print("\n=== Incompatible Features ===")
print(full_results['incompatible_features'])

=== Numeric Features ===

age:
  psi: 0.0846
  kl: 0.0423
  ks: stat=0.1273, p=0.0000
  js: 0.0105
  wd: 2.8674

income:
  psi: 0.2688
  kl: 0.1324
  ks: stat=0.2105, p=0.0000
  js: 0.0328
  wd: 7527.0382

=== Categorical Features ===

gender:
  psi: 0.0392
  kl: 0.0197
  js: 0.0049
  wd: 0.0985

education:
  psi: 0.0330
  kl: 0.0166
  js: 0.0041
  wd: 0.0825

=== Incompatible Features ===
[]


## 7. Individual feature metric computations

In [18]:
# Compute PSI for a single numeric feature
psi_age = detector3.compute_feature_psi('age')
print(f"PSI for age: {psi_age:.4f}")

# Compute KL divergence for a categorical feature
kl_gender = detector3.compute_feature_kl('gender')
print(f"KL for gender: {kl_gender:.4f}")

# Compute KS for a numeric feature (returns dict with statistic and pvalue)
ks_income = detector3.compute_feature_ks('income')
print(f"KS for income: statistic={ks_income['statistic']:.4f}, pvalue={ks_income['pvalue']:.4f}")

# Compute JS divergence
js_education = detector3.compute_feature_js('education')
print(f"JS for education: {js_education:.4f}")

# Compute Wasserstein distance
wd_income = detector3.compute_feature_wd('income')
print(f"Wasserstein distance for income: {wd_income:.4f}")

PSI for age: 0.0846
KL for gender: 0.0197
KS for income: statistic=0.2105, pvalue=0.0000
JS for education: 0.0041
Wasserstein distance for income: 7527.0382


## 8. Handling incompatible types (force a type mismatch)

In [19]:
# Create a copy and change dtype of a column in current set
ref_mismatch = ref_df.copy()
cur_mismatch = cur_df.copy()
cur_mismatch['age'] = cur_mismatch['age'].astype(str)   # convert numeric to string

detector_mismatch = DataDriftDetector(ref_mismatch, cur_mismatch, features=['age', 'gender'])

print("Numeric features after mismatch:", detector_mismatch.numeric_features)
print("Categorical features after mismatch:", detector_mismatch.categorical_features)
print("Incompatible features:", detector_mismatch.incompatible_features)
# The warning about incompatible types will appear when initialising.

Numeric features after mismatch: []
Categorical features after mismatch: ['gender']
Incompatible features: ['age']


E:\git repo\momo_ml\momo_ml\monitor\data_drift.py:85: UserWarning: Feature 'age' has incompatible types: ref type = float64, cur type = str. It will be excluded from drift detection.
  warnings.warn(


## 9. Customising KL / JS parameters

In [20]:
# You can adjust binning and base for KL/JS via constructor arguments
detector_custom = DataDriftDetector(
    ref_df, cur_df,
    kl_buckets=20,          # number of bins for continuous features
    kl_base='2',            # use base 2 (bits)
    kl_epsilon=1e-10,
    kl_handle_outside='ignore'   # how to handle values outside reference range
)

# Compute KL for a numeric feature with these settings
kl_custom = detector_custom.compute_feature_kl('income')
print(f"KL (custom settings) for income: {kl_custom:.4f}")

KL (custom settings) for income: 0.2086


The `DataDriftDetector` provides a unified interface to monitor feature drift between two datasets.  
Key points:

- Features are automatically split into numeric and categorical based on dtypes.
- PSI, KL, JS, and Wasserstein distance are available for both types; KS is numeric‑only.
- You can specify a feature list; missing features or incompatible types issue warnings.
- Individual feature metrics can be accessed via `compute_feature_*` methods.
- The `compute()` method returns a structured dictionary with all results.

These metrics help identify distribution shifts that may affect model performance.